# NB06 — Stocks as a Market Signal
**Global Oil Market Analysis**

Oil inventories are the oil market's buffer. When production exceeds consumption, the surplus goes into storage. Traders watch stock levels closely because high inventories signal oversupply and future price pressure. The EIA publishes US inventory data every Wednesday — it is one of the most watched weekly data releases in financial markets. This notebook tests whether stock levels actually predict price direction and examines the role of the US Strategic Petroleum Reserve (SPR) as a policy tool.

---

## Sections
1. US commercial crude stocks vs price
2. Does high inventory predict falling prices?
3. The Strategic Petroleum Reserve — policy tool or market signal?
4. Global stocks vs price (2012-2025)
5. Regional US inventory — Cushing and the WTI benchmark

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.2f}'.format)

DATA_DIR = './data/processed/'

master       = pd.read_parquet(DATA_DIR + 'master_actuals.parquet')
us_stocks    = pd.read_parquet(DATA_DIR + 'us_stocks.parquet')
glob_stocks  = pd.read_parquet(DATA_DIR + 'global_stocks.parquet')
prices       = pd.read_parquet(DATA_DIR + 'prices.parquet')

# Also load raw weekly US stocks for higher-frequency analysis
import openpyxl
def load_excel_sheet(filepath, sheet_name, col_names=None):
    wb = openpyxl.load_workbook(filepath, read_only=True, data_only=True)
    ws = wb[sheet_name]
    rows, headers = [], None
    for i, row in enumerate(ws.iter_rows(values_only=True)):
        if i == 0: continue
        if i == 1:
            headers = ['date_str','date'] + (col_names or [str(c) for c in row[2:]])
            continue
        rows.append(list(row))
    wb.close()
    df = pd.DataFrame(rows, columns=headers)
    df['date'] = pd.to_datetime(df['date'])
    df = df.replace('NA', np.nan).drop(columns=['date_str']).dropna(subset=['date']).set_index('date')
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

us_stocks_weekly = load_excel_sheet(
    './data/oil-stocks-in-the-us-eia.xlsx', 'Oil stocks',
    col_names=['us_stocks_excl_spr', 'us_stocks_spr', 'us_stocks_total']
)
# Convert to million barrels
us_stocks_weekly = us_stocks_weekly / 1000

print('Data loaded.')
print(f'US stocks (weekly): {us_stocks_weekly.shape}')
print(f'Global stocks     : {glob_stocks.shape}')

Data loaded.
US stocks (weekly): (2267, 3)
Global stocks     : (168, 1)


## 1. US Commercial Crude Stocks vs Price

In [2]:
# Join monthly US stocks with monthly price
us_price = us_stocks.join(prices[['price_world']], how='inner').dropna()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=['US Commercial Crude Stocks excl. SPR (million barrels)',
                                    'World Oil Price (USD/bbl)'])

fig.add_trace(go.Scatter(
    x=us_price.index, y=us_price['us_stocks_excl_spr_mmbbl'],
    mode='lines', name='US commercial stocks',
    fill='tozeroy', fillcolor='rgba(5,150,105,0.1)',
    line=dict(color='#059669', width=2)
), row=1, col=1)

# 5-year average band (rough normal range)
rolling_mean = us_price['us_stocks_excl_spr_mmbbl'].rolling(60).mean()
rolling_std  = us_price['us_stocks_excl_spr_mmbbl'].rolling(60).std()
fig.add_trace(go.Scatter(
    x=us_price.index, y=rolling_mean + rolling_std,
    mode='lines', showlegend=False,
    line=dict(color='gray', width=0.5, dash='dot')
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=us_price.index, y=rolling_mean - rolling_std,
    mode='lines', name='5-yr normal range',
    fill='tonexty', fillcolor='rgba(156,163,175,0.15)',
    line=dict(color='gray', width=0.5, dash='dot')
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=us_price.index, y=us_price['price_world'],
    mode='lines', name='Price',
    line=dict(color='#DC2626', width=2)
), row=2, col=1)

fig.update_layout(
    title='US Commercial Crude Stocks vs Oil Price',
    template='plotly_white', height=560, hovermode='x unified',
    legend=dict(orientation='h', y=-0.15)
)
fig.show()

## 2. Does High Inventory Predict Falling Prices?

In [3]:
# Test: does stock level predict next 3-month price direction?
test = us_price.copy()
test['stock_z_score'] = (
    (test['us_stocks_excl_spr_mmbbl'] - test['us_stocks_excl_spr_mmbbl'].rolling(60).mean())
    / test['us_stocks_excl_spr_mmbbl'].rolling(60).std()
)
test['price_3m_chg'] = test['price_world'].pct_change(3).shift(-3) * 100

clean_test = test.dropna(subset=['stock_z_score', 'price_3m_chg'])
r, p = stats.pearsonr(clean_test['stock_z_score'], clean_test['price_3m_chg'])

print(f'Correlation: US stock z-score vs next 3-month price change')
print(f'  Pearson r = {r:.3f}')
print(f'  p-value   = {p:.4f}')
print(f'  Significant at p<0.05: {p < 0.05}')
print()
print('Interpretation:')
if r < -0.1 and p < 0.05:
    print('  High inventories predict falling prices — stocks work as a leading indicator.')
elif abs(r) < 0.1 or p >= 0.05:
    print('  Weak relationship — US stocks are not a reliable price predictor at this frequency.')
else:
    print('  Unexpected direction — prices and stocks move together in this sample.')

Correlation: US stock z-score vs next 3-month price change
  Pearson r = 0.182
  p-value   = 0.0001
  Significant at p<0.05: True

Interpretation:
  Unexpected direction — prices and stocks move together in this sample.


In [4]:
# Scatter: stock z-score vs 3-month price change
fig = px.scatter(
    clean_test, x='stock_z_score', y='price_3m_chg',
    trendline='ols',
    color='price_3m_chg',
    color_continuous_scale='RdYlGn',
    labels={
        'stock_z_score': 'US Commercial Stocks (z-score vs 5-yr avg)',
        'price_3m_chg': 'Price change 3 months later (%)'
    },
    title=f'US Stock Level vs Next 3-Month Price Change (r = {r:.3f}, p = {p:.4f})'
)
fig.add_hline(y=0, line_dash='dash', line_color='gray', opacity=0.5)
fig.add_vline(x=0, line_dash='dash', line_color='gray', opacity=0.5)
fig.add_annotation(
    x=-2, y=-25,
    text='Low stocks, prices rose',
    showarrow=False, font=dict(size=10, color='green')
)
fig.add_annotation(
    x=1.5, y=20,
    text='High stocks, prices fell',
    showarrow=False, font=dict(size=10, color='red')
)
fig.update_layout(template='plotly_white', height=500, showlegend=False)
fig.show()

## 3. The Strategic Petroleum Reserve — Policy Tool

In [5]:
# SPR history — created 1975 after the Arab oil embargo
spr = us_stocks_weekly[['us_stocks_spr']].dropna()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=spr.index, y=spr['us_stocks_spr'],
    mode='lines', name='SPR (million barrels)',
    fill='tozeroy', fillcolor='rgba(124,58,237,0.1)',
    line=dict(color='#7C3AED', width=2)
))

# Major SPR releases
spr_events = [
    ('1990-08', 'Gulf War (small release)', '#D97706'),
    ('2005-09', 'Hurricane Katrina',        '#DC2626'),
    ('2011-06', 'Libya supply disruption',  '#DC2626'),
    ('2022-03', 'Ukraine war — record\n180M bbl release', '#DC2626'),
]
for dt, label, color in spr_events:
    fig.add_vline(x=dt, line_dash='dash', line_color=color, opacity=0.6)
    spr_val = spr.loc[spr.index >= pd.Timestamp(dt)].iloc[0]['us_stocks_spr'] if not spr.loc[spr.index >= pd.Timestamp(dt)].empty else 400
    fig.add_annotation(
        x=pd.Timestamp(dt), y=spr_val + 30,
        text=label, showarrow=False,
        textangle=-90, font=dict(size=9, color=color),
        xanchor='left'
    )

fig.update_layout(
    title='US Strategic Petroleum Reserve (million barrels) — 1982 to 2026',
    xaxis_title='Date', yaxis_title='SPR (million barrels)',
    template='plotly_white', height=450
)
fig.show()

In [6]:
spr_clean = spr.dropna()
print('SPR KEY STATISTICS:')
print(f'  Peak level    : {spr_clean["us_stocks_spr"].max():.0f} M bbl ({spr_clean["us_stocks_spr"].idxmax().date()})')
print(f'  Current level : {spr_clean["us_stocks_spr"].iloc[-1]:.0f} M bbl ({spr_clean.index[-1].date()})')
drawdown = spr_clean['us_stocks_spr'].max() - spr_clean['us_stocks_spr'].iloc[-1]
drawdown_pct = drawdown / spr_clean['us_stocks_spr'].max() * 100
print(f'  Drawn down    : {drawdown:.0f} M bbl ({drawdown_pct:.1f}% from peak)')
print(f'  At peak fill  : {spr_clean["us_stocks_spr"].idxmax().date()}')
print()
print('The 2022 release was the largest in SPR history — 180M barrels over 6 months.')

SPR KEY STATISTICS:
  Peak level    : 727 M bbl (2010-01-01)
  Current level : 415 M bbl (2026-03-06)
  Drawn down    : 311 M bbl (42.8% from peak)
  At peak fill  : 2010-01-01

The 2022 release was the largest in SPR history — 180M barrels over 6 months.


## 4. Global Stocks vs Price (2012-2025)

In [7]:
glob_price = glob_stocks.join(prices[['price_world']], how='inner').dropna()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=['Global Crude Stocks (million barrels)',
                                    'World Oil Price (USD/bbl)'])

fig.add_trace(go.Scatter(
    x=glob_price.index, y=glob_price['global_stocks_mmbbl'],
    mode='lines', name='Global stocks',
    fill='tozeroy', fillcolor='rgba(5,150,105,0.1)',
    line=dict(color='#059669', width=2)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=glob_price.index, y=glob_price['price_world'],
    mode='lines', name='Price',
    line=dict(color='#DC2626', width=2)
), row=2, col=1)

fig.update_layout(
    title='Global Crude Stocks vs Oil Price (2012-2025)',
    template='plotly_white', height=560, hovermode='x unified',
    showlegend=False
)
fig.show()

In [8]:
# Correlation: global stocks vs price
r_glob, p_glob = stats.pearsonr(glob_price['global_stocks_mmbbl'], glob_price['price_world'])
print(f'Global stocks vs price correlation: r = {r_glob:.3f} (p = {p_glob:.4f})')

# Global stocks vs next 3-month price change
glob_price['price_3m_chg'] = glob_price['price_world'].pct_change(3).shift(-3) * 100
clean_glob = glob_price.dropna(subset=['global_stocks_mmbbl', 'price_3m_chg'])
r_fwd, p_fwd = stats.pearsonr(clean_glob['global_stocks_mmbbl'], clean_glob['price_3m_chg'])
print(f'Global stocks vs NEXT 3-month price change: r = {r_fwd:.3f} (p = {p_fwd:.4f})')

Global stocks vs price correlation: r = -0.327 (p = 0.0000)
Global stocks vs NEXT 3-month price change: r = 0.081 (p = 0.3030)


In [11]:
# Final summary across all notebooks
print('=== NB06 KEY FINDINGS ===')
print()
print('US Commercial Stocks:')
us_min  = us_stocks_weekly['us_stocks_excl_spr'].dropna().min()
us_max  = us_stocks_weekly['us_stocks_excl_spr'].dropna().max()
us_late = us_stocks_weekly['us_stocks_excl_spr'].dropna().iloc[-1]
print(f'  Historical range: {us_min:.0f} to {us_max:.0f} million barrels')
print(f'  Latest reading  : {us_late:.0f} million barrels')
print()
print('Strategic Petroleum Reserve:')
print(f'  Peak: {spr_clean["us_stocks_spr"].max():.0f} M bbl | Current: {spr_clean["us_stocks_spr"].iloc[-1]:.0f} M bbl')
print()
print('Predictive power of stocks for prices:')
print(f'  US commercial stocks vs next 3-month price: r = {r:.3f} (p = {p:.4f})')
print(f'  Global stocks vs next 3-month price: r = {r_fwd:.3f} (p = {p_fwd:.4f})')
print()
print('All notebooks complete.')

=== NB06 KEY FINDINGS ===

US Commercial Stocks:
  Historical range: 247 to 541 million barrels
  Latest reading  : 443 million barrels

Strategic Petroleum Reserve:
  Peak: 727 M bbl | Current: 415 M bbl

Predictive power of stocks for prices:
  US commercial stocks vs next 3-month price: r = 0.182 (p = 0.0001)
  Global stocks vs next 3-month price: r = 0.081 (p = 0.3030)

All notebooks complete.
